In [0]:
%run /Workspace/Users/riley.day@kwa-analytics.com/Databricks-Certified-Data-Engineer-Associate/Includes/Copy-Datasets

In [0]:
(spark.readStream
    .table("books")
    .createOrReplaceTempView("books_streaming_tmp_vw"))

In [0]:
%sql
SELECT * FROM books_streaming_tmp_vw

In [0]:
%sql
SELECT author, count(book_id) as total_books
from books_streaming_tmp_vw
group by author

In [0]:
%sql
SELECT author, count(book_id) as total_books
from books_streaming_tmp_vw
group by author
order by author

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW author_counts_tmp_vw AS (
    SELECT author, count(book_id) as total_books
    from books_streaming_tmp_vw
    group by author
)

In [0]:
(spark.table("author_counts_tmp_vw")
 .writeStream
 .trigger(processingTime="4 seconds")
 .outputMode("Complete")
 .option("checkpointLocation", "dbfs:/mnt/demo/author_counts_checkpoint")
 .table("author_counts")
 )

In [0]:
%sql
select * from author_counts

In [0]:
%sql
INSERT INTO books
values 
    ("B19", "Introduction to Modeling and Simulation", "Mark W. Spong", 25),
    ("B20", "Robot Modeling and Control", "Mark W. Spong", 30),
    ("B21", "Turing's Vision: The Birth of Computer SCience", "Chris Bernhardt", 35)


In [0]:
%sql
select * from author_counts

In [0]:
%sql
INSERT INTO books
values 
    ("B16", "Hands-On Deep Learning Algorithms with Python", "Sudharsan Ravichandiran", 25),
    ("B17", "Neural Network Methods in Natural Language Processing", "Yoav Goldberg", 30),
    ("B18", "Understanding Digital Signal Processing", "Richard Lyons", 35)


In [0]:
(spark.table("author_counts_tmp_vw")
 .writeStream
 .trigger(availableNow=True)
 .outputMode("Complete")
 .option("checkpointLocation", "dbfs:/mnt/demo/author_counts_checkpoint")
 .table("author_counts")
 .awaitsTermination()
 )

In [0]:
%sql
select * from author_counts